<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_TransmissionExpansion_LinearDC_OPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# INSTALL
# =====================================================
!pip -q install pyomo highspy pypower openpyxl pandas numpy

# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import pyomo.environ as pyo

from pypower.idx_bus import BUS_I, PD
from pypower.idx_gen import GEN_BUS, PMIN, PMAX
from pypower.idx_brch import F_BUS, T_BUS, BR_X, BR_STATUS
from pypower.makeBdc import makeBdc


# =====================================================
# READ YOUR EXACT EXCEL FILE
# =====================================================
EXCEL = "/content/Data6busSystemTEP.xlsx"
BASE_MVA = 100
SLACK_BUS = 1


def build_ppc():

    gen_df  = pd.read_excel(EXCEL,"generator")
    load_df = pd.read_excel(EXCEL,"load")
    line_df = pd.read_excel(EXCEL,"lines")

    # -----------------------------
    # INTERNAL BUS INDEX (0-based)
    # -----------------------------
    buses = sorted(
        set(gen_df["generator bus"])
        | set(load_df["load bus"])
        | set(line_df["from"])
        | set(line_df["to"])
    )

    ext2int = {b:i for i,b in enumerate(buses)}
    int2ext = {i:b for b,i in ext2int.items()}

    nb=len(buses)

    # -----------------------------
    # BUS MATRIX
    # -----------------------------
    bus=np.zeros((nb,13))

    for i in range(nb):
        bus[i,BUS_I]=i

    for _,r in load_df.iterrows():
        i=ext2int[r["load bus"]]
        bus[i,PD]+=r["Pload"]

    # -----------------------------
    # GENERATORS
    # -----------------------------
    ng=len(gen_df)
    gen=np.zeros((ng,10))
    cost=np.zeros((ng,3))

    for k,r in gen_df.iterrows():

        gen[k,GEN_BUS]=ext2int[r["generator bus"]]
        gen[k,PMIN]=r["Pmin"]
        gen[k,PMAX]=r["Pmax"]

        cost[k]=[r["a"],r["b"],r["c"]]

    # -----------------------------
    # BRANCH
    # -----------------------------
    nl=len(line_df)
    branch=np.zeros((nl,13))

    for k,r in line_df.iterrows():

        branch[k,F_BUS]=ext2int[r["from"]]
        branch[k,T_BUS]=ext2int[r["to"]]
        branch[k,BR_X]=r["X"]
        branch[k,BR_STATUS]=1   # ⭐ CRITICAL

    return {
        "bus":bus,
        "gen":gen,
        "branch":branch,
        "cost":cost,
        "ext2int":ext2int,
        "int2ext":int2ext,
        "slack":ext2int[SLACK_BUS],
        "nb":nb,
        "ng":ng
    }


# =====================================================
# DC MATRICES
# =====================================================
def dc_matrices(ppc):

    Bbus,Bf,Pbusinj,_=makeBdc(
        BASE_MVA,
        ppc["bus"],
        ppc["branch"]
    )

    return np.array(Bbus.todense())


# =====================================================
# LP DC-OPF (Pyomo + HiGHS)
# =====================================================
def solve_dc_opf(ppc):

    Bbus=dc_matrices(ppc)

    bus=ppc["bus"]
    gen=ppc["gen"]
    a,b,c=ppc["cost"].T

    nb=ppc["nb"]
    ng=ppc["ng"]
    slack=ppc["slack"]

    Pd=bus[:,PD]
    gen_bus=gen[:,GEN_BUS].astype(int)

    Pmin=gen[:,PMIN]
    Pmax=gen[:,PMAX]

    print("TOTAL LOAD =",Pd.sum(),"MW")
    print("TOTAL PMAX =",Pmax.sum(),"MW")

    m=pyo.ConcreteModel()

    m.B=pyo.RangeSet(0,nb-1)
    m.G=pyo.RangeSet(0,ng-1)

    m.theta=pyo.Var(m.B)
    m.Pg=pyo.Var(m.G)

    m.theta[slack].fix(0)

    # limits
    def lim_rule(mdl,g):
        return (Pmin[g],mdl.Pg[g],Pmax[g])
    m.lim=pyo.Constraint(m.G,rule=lim_rule)

    # nodal balance
    def balance(mdl,i):

        Pg=sum(
            mdl.Pg[g]
            for g in mdl.G
            if gen_bus[g]==i
        )

        network=BASE_MVA*sum(
            Bbus[i,j]*mdl.theta[j]
            for j in mdl.B
        )

        return Pg-Pd[i]==network

    m.balance=pyo.Constraint(m.B,rule=balance)

    # quadratic → linear approximation
    m.obj=pyo.Objective(
        expr=sum(b[g]*m.Pg[g] for g in m.G)
    )

    solver=pyo.SolverFactory("highs")
    res=solver.solve(m)

    Pg=np.array([pyo.value(m.Pg[g]) for g in m.G])
    theta=np.array([pyo.value(m.theta[i]) for i in m.B])

    return Pg,theta


# =====================================================
# RUN
# =====================================================
ppc=build_ppc()

Pg,theta=solve_dc_opf(ppc)

print("\nDISPATCH:")
for g,val in enumerate(Pg):
    bus=ppc["int2ext"][int(ppc["gen"][g,GEN_BUS])]
    print(f"Gen at bus {bus}: {val:.2f} MW")

print("\nANGLES:")
for i,val in enumerate(theta):
    print(f"Bus {ppc['int2ext'][i]} : {val:.5f}")

# =====================================================
# POST-PROCESSING RESULTS
# =====================================================

def compute_results(ppc, Pg, theta):

    BASE_MVA = 100

    gen = ppc["gen"]
    cost = ppc["cost"]
    branch = ppc["branch"]
    int2ext = ppc["int2ext"]

    a,b,c = cost.T

    # ===============================
    # GENERATOR TABLE
    # ===============================
    gen_results = []

    total_cost = 0

    for g,P in enumerate(Pg):

        C = a[g] + b[g]*P + c[g]*P**2
        total_cost += C

        bus_ext = int2ext[int(gen[g,GEN_BUS])]

        gen_results.append([
            g,
            bus_ext,
            P,
            C
        ])

    gen_table = pd.DataFrame(
        gen_results,
        columns=[
            "Generator",
            "Bus",
            "Pg (MW)",
            "Cost ($/h)"
        ]
    )

    # ===============================
    # LINE FLOWS
    # ===============================
    flow_results=[]

    for k in range(len(branch)):

        i=int(branch[k,F_BUS])
        j=int(branch[k,T_BUS])
        x=branch[k,BR_X]

        flow = BASE_MVA*(theta[i]-theta[j])/x

        flow_results.append([
            int2ext[i],
            int2ext[j],
            flow
        ])

    flow_table=pd.DataFrame(
        flow_results,
        columns=[
            "From Bus",
            "To Bus",
            "Power Flow (MW)"
        ]
    )

    # ===============================
    # BUS ANGLES
    # ===============================
    angle_table=pd.DataFrame({
        "Bus":[int2ext[i] for i in range(len(theta))],
        "Theta (rad)":theta
    })

    return gen_table, flow_table, angle_table, total_cost

gen_table, flow_table, angle_table, total_cost = \
    compute_results(ppc, Pg, theta)

print("\n===== GENERATOR DISPATCH =====")
display(gen_table)

print("\n===== LINE FLOWS =====")
display(flow_table)

print("\n===== BUS ANGLES =====")
display(angle_table)

print("\nTOTAL SYSTEM COST = %.2f $/h" % total_cost)

TOTAL LOAD = 300.0 MW
TOTAL PMAX = 530.0 MW

DISPATCH:
Gen at bus 1: 50.00 MW
Gen at bus 2: 150.00 MW
Gen at bus 3: 100.00 MW

ANGLES:
Bus 1 : 0.00000
Bus 2 : 0.01309
Bus 3 : 0.00797
Bus 4 : -0.06086
Bus 5 : -0.07835
Bus 6 : -0.06087

===== GENERATOR DISPATCH =====


,Generator,Bus,Pg (MW),Cost ($/h)
0,0,1,50.0,809.875
1,1,2,150.0,1949.975
2,2,3,100.0,1397.400



===== LINE FLOWS =====


,From Bus,To Bus,Power Flow (MW)
0,1,2,-6.545120
1,1,4,30.427906
2,1,5,26.117214
3,2,3,2.046936
4,2,4,73.946052
5,2,5,30.480628
6,2,6,36.981264
7,3,5,33.201748
8,3,6,68.845188
9,4,5,4.373958



===== BUS ANGLES =====


,Bus,Theta (rad)
0,1,0.000000
1,2,0.013090
2,3,0.007973
3,4,-0.060856
4,5,-0.078352
5,6,-0.060872



TOTAL SYSTEM COST = 4157.25 $/h
